In [1]:
!python --version

Python 3.14.7


In [11]:
!pip install matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 10.5 MB/s  0:00:000.6 MB/s eta 0:00:01:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 8.4 MB/s  0:00:00m 10.3 MB/s eta 0:00:01
Using cached packaging-26.3-py3-none-any.whl (129 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 8.0 MB/s  0:00:007.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [matplotlib] 7/8 [matplotlib]


In [ ]:
import pandas as pd

FILE_PATH = "***"

usecols = [
    "event_time",
    "event_type",
    "product_id",
    "category_code",
    "price",
    "user_id"
]

# 最初の100万行だけ読む
df = pd.read_csv(
    FILE_PATH,
    usecols=usecols,
    nrows=1_000_000
)

# view / purchaseだけ
df = df[
    df["event_type"].isin(["view", "purchase"])
].copy()

# 最初の10ユーザーを取得
target_users = df["user_id"].dropna().unique()[:10000]

df = df[
    df["user_id"].isin(target_users)
].copy()

# timestamp
df["event_time"] = pd.to_datetime(df["event_time"])

df = df.rename(columns={
    "event_time": "timestamp",
    "category_code": "genre_path"
})

df = df.sort_values(
    ["user_id", "timestamp"]
).reset_index(drop=True)

print(df.shape)
df.head()

(71402, 6)


,timestamp,event_type,product_id,genre_path,price,user_id
0,2019-11-01 02:23:03+00:00,view,1306265,computers.notebook,1415.48,275256741
1,2019-11-01 01:43:56+00:00,view,1005196,electronics.smartphone,431.41,389979783
2,2019-11-01 01:44:00+00:00,view,1005196,electronics.smartphone,431.41,389979783
3,2019-11-01 01:44:13+00:00,view,1005196,electronics.smartphone,431.41,389979783
4,2019-11-01 01:44:47+00:00,view,1005196,electronics.smartphone,431.41,389979783


In [15]:
from node2vec import Node2Vec

node2vec = Node2Vec(
    G,
    dimensions=64,
    walk_length=20,
    num_walks=50,
    workers=4,
    seed=42
)

model = node2vec.fit(
    window=5,
    min_count=1,
    batch_words=128
)

EVENT_WEIGHT = {
    "view": 1.0,
    "purchase": 5.0
}

import numpy as np

def make_user_embedding(user_df, model):
    vectors = []
    weights = []

    for _, row in user_df.iterrows():
        category = row["genre_path"]
        event_type = row["event_type"]

        # categoryがNode2Vecに存在しない場合
        if category not in model.wv:
            continue

        # view=1, purchase=5
        weight = EVENT_WEIGHT.get(event_type, 1.0)

        vectors.append(model.wv[category])
        weights.append(weight)

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    vectors = np.array(vectors)
    weights = np.array(weights)

    # 重み付き平均
    return np.average(
        vectors,
        axis=0,
        weights=weights
    )

user_ids = []
user_embeddings = []

for user_id, user_df in df.groupby("user_id"):

    embedding = make_user_embedding(
        user_df,
        model
    )

    user_ids.append(user_id)
    user_embeddings.append(embedding)

X = np.vstack(user_embeddings)

print(X.shape)

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init="auto"
)

clusters = kmeans.fit_predict(X_scaled)

user_cluster_df = pd.DataFrame({
    "user_id": user_ids,
    "cluster": clusters
})

user_cluster_df.head(100)

Generating walks (CPU: 4): 100%|██████████| 12/12 [00:00<00:00, 125.20it/s]


(7452, 64)


,user_id,cluster
0,275256741,0
1,389979783,1
2,415485440,1
3,422713919,2
4,434890838,2
...,...,...
95,512421287,0
96,512422125,4
97,512423283,0
98,512423604,1


In [4]:
import pandas as pd

df = df.dropna(subset=["genre_path"])
df = df.sort_values(["user_id", "timestamp"])

rows = []

for user_id, user_df in df.groupby("user_id"):

    browser_history = []
    browser_genre_history = []

    for _, row in user_df.iterrows():

        if row["event_type"] == "view":
            browser_history.append(row["product_id"])
            browser_genre_history.append(row["genre_path"])

        elif row["event_type"] == "purchase":
            rows.append({
                "user_id": user_id,
                "timestamp": row["timestamp"],
                "genre_path": row["genre_path"],
                "price": row["price"],
                "product_id": row["product_id"],
                "browser_history": browser_history.copy(),
                "browser_genre_history": browser_genre_history.copy()
            })

            browser_history = []
            browser_genre_history = []

purchase_df = pd.DataFrame(rows)

print(len(purchase_df))

purchase_df.head()

717


,user_id,timestamp,genre_path,price,product_id,browser_history,browser_genre_history
0,486999716,2019-11-01 00:34:18+00:00,electronics.smartphone,242.72,1004768,[1004768],[electronics.smartphone]
1,512382878,2019-11-01 02:45:00+00:00,electronics.video.tv,228.76,1801805,"[1801805, 1801805]","[electronics.video.tv, electronics.video.tv]"
2,512400164,2019-11-01 01:50:43+00:00,electronics.smartphone,242.72,1004766,[1004766],[electronics.smartphone]
3,512405571,2019-11-01 08:00:31+00:00,electronics.smartphone,431.41,1005196,"[28703869, 28703986, 28704348, 28704348, 28703...","[apparel.shoes.keds, apparel.shoes.keds, appar..."
4,512452645,2019-11-01 02:12:21+00:00,electronics.smartphone,577.89,1003306,[1003306],[electronics.smartphone]


In [16]:
import numpy as np

def browse_to_embedding(history, model):
    vectors = [
        model.wv[category]
        for category in history
        if category in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

purchase_with_browse_df = purchase_df[
    purchase_df["browser_genre_history"].apply(len) > 0
].copy()

print("all purchases:", len(purchase_df))
print("purchases with browse:", len(purchase_with_browse_df))

X = np.vstack(
    purchase_with_browse_df["browser_genre_history"]
    .apply(lambda x: browse_to_embedding(x, model))
)

print(X.shape)

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init="auto"
)

purchase_with_browse_df["browse_cluster"] = (
    kmeans.fit_predict(X_scaled)
)

from collections import Counter

for cluster_id in sorted(
    purchase_with_browse_df["browse_cluster"].unique()
):
    cluster_df = purchase_with_browse_df[
        purchase_with_browse_df["browse_cluster"] == cluster_id
    ]

    browse_categories = []

    for history in cluster_df["browser_genre_history"]:
        browse_categories.extend(history)

    print(f"\n===== Cluster {cluster_id} =====")
    print("samples:", len(cluster_df))
    print("Top browsed categories:")

    for category, count in Counter(browse_categories).most_common(10):
        print(category, count)

all purchases: 717
purchases with browse: 713
(713, 64)

===== Cluster 0 =====
samples: 536
Top browsed categories:
electronics.smartphone 1974
electronics.audio.headphone 209
electronics.video.tv 140
construction.tools.drill 75
electronics.clocks 71
construction.components.faucet 54
accessories.bag 38
accessories.wallet 36
computers.notebook 31
electronics.telephone 28

===== Cluster 1 =====
samples: 83
Top browsed categories:
appliances.kitchen.washer 188
appliances.environment.vacuum 166
appliances.kitchen.blender 63
appliances.kitchen.refrigerators 46
appliances.sewing_machine 44
appliances.environment.water_heater 34
appliances.kitchen.microwave 23
appliances.kitchen.mixer 21
electronics.smartphone 17
appliances.kitchen.oven 11

===== Cluster 2 =====
samples: 41
Top browsed categories:
computers.desktop 218
computers.notebook 76
computers.components.videocards 18
computers.peripherals.mouse 16
computers.peripherals.printer 15
computers.components.hdd 11
computers.peripherals.monit

In [17]:
for cluster_id in sorted(
    purchase_with_browse_df["browse_cluster"].unique()
):
    cluster_df = purchase_with_browse_df[
        purchase_with_browse_df["browse_cluster"] == cluster_id
    ]

    print(f"\n===== Cluster {cluster_id} =====")
    print("samples:", len(cluster_df))

    print("\nTop PURCHASE categories:")
    print(
        cluster_df["genre_path"]
        .value_counts()
        .head(10)
    )


===== Cluster 0 =====
samples: 536

Top PURCHASE categories:
genre_path
electronics.smartphone            373
electronics.audio.headphone        65
electronics.video.tv               32
electronics.clocks                 17
electronics.tablet                  9
electronics.telephone               7
electronics.audio.subwoofer         5
accessories.bag                     4
accessories.wallet                  3
construction.components.faucet      2
Name: count, dtype: int64

===== Cluster 1 =====
samples: 83

Top PURCHASE categories:
genre_path
appliances.kitchen.washer              22
appliances.environment.vacuum          13
appliances.kitchen.refrigerators        8
appliances.environment.water_heater     8
appliances.kitchen.blender              6
appliances.kitchen.microwave            5
appliances.sewing_machine               2
appliances.kitchen.oven                 2
appliances.kitchen.coffee_machine       2
appliances.kitchen.hob                  2
Name: count, dtype: int64

==

In [18]:
product_ranking = (
    purchase_with_browse_df
    .groupby(["browse_cluster", "product_id"])
    .size()
    .reset_index(name="purchase_count")
)

product_ranking["purchase_ratio"] = (
    product_ranking["purchase_count"]
    / product_ranking
        .groupby("browse_cluster")["purchase_count"]
        .transform("sum")
)

product_ranking = product_ranking.sort_values(
    ["browse_cluster", "purchase_count"],
    ascending=[True, False]
)

for cluster_id in range(5):
    print(f"\n===== Cluster {cluster_id}: Recommended Products =====")

    display(
        product_ranking[
            product_ranking["browse_cluster"] == cluster_id
        ].head(10)
    )


===== Cluster 0: Recommended Products =====


,browse_cluster,product_id,purchase_count,purchase_ratio
58,0,1004856,38,0.070896
162,0,4804056,22,0.041045
91,0,1005115,18,0.033582
43,0,1004767,16,0.029851
52,0,1004833,14,0.026119
61,0,1004870,14,0.026119
38,0,1004739,13,0.024254
22,0,1004249,12,0.022388
40,0,1004750,12,0.022388
88,0,1005105,11,0.020522



===== Cluster 1: Recommended Products =====


,browse_cluster,product_id,purchase_count,purchase_ratio
265,1,3900002,6,0.072289
232,1,3100034,3,0.036145
241,1,3600661,3,0.036145
222,1,2702277,2,0.024096
237,1,3600163,2,0.024096
239,1,3600452,2,0.024096
244,1,3601124,2,0.024096
251,1,3601524,2,0.024096
255,1,3700338,2,0.024096
256,1,3700766,2,0.024096



===== Cluster 2: Recommended Products =====


,browse_cluster,product_id,purchase_count,purchase_ratio
299,2,1701474,3,0.073171
284,2,1307074,2,0.048780
304,2,6701117,2,0.048780
313,2,17800113,2,0.048780
314,2,17800248,2,0.048780
282,2,1306256,1,0.024390
283,2,1307073,1,0.024390
285,2,1307078,1,0.024390
286,2,1307135,1,0.024390
287,2,1307307,1,0.024390



===== Cluster 3: Recommended Products =====


,browse_cluster,product_id,purchase_count,purchase_ratio
328,3,28717081,3,0.157895
317,3,1004856,1,0.052632
318,3,1801881,1,0.052632
319,3,25800007,1,0.052632
320,3,28704386,1,0.052632
321,3,28704601,1,0.052632
322,3,28715623,1,0.052632
323,3,28716422,1,0.052632
324,3,28716454,1,0.052632
325,3,28716509,1,0.052632



===== Cluster 4: Recommended Products =====


,browse_cluster,product_id,purchase_count,purchase_ratio
353,4,14700139,6,0.176471
350,4,13201206,3,0.088235
334,4,1002544,1,0.029412
335,4,1002545,1,0.029412
336,4,1004838,1,0.029412
337,4,1005105,1,0.029412
338,4,4000170,1,0.029412
339,4,5700785,1,0.029412
340,4,5701066,1,0.029412
341,4,5701246,1,0.029412


In [5]:
import networkx as nx

G = nx.Graph()

categories = df["genre_path"].dropna().unique()

for category in categories:

    parts = category.split(".")

    # electronics.video.tv
    # ↓
    # electronics
    # electronics.video
    # electronics.video.tv

    nodes = [
        ".".join(parts[:i])
        for i in range(1, len(parts) + 1)
    ]

    # parent-child edge
    for parent, child in zip(nodes[:-1], nodes[1:]):
        G.add_edge(parent, child)

print("nodes:", G.number_of_nodes())
print("edges:", G.number_of_edges())

nodes: 151
edges: 138


In [6]:
import numpy as np

def history_to_embedding(history, model):

    vectors = []

    for category in history:

        if category in model.wv:
            vectors.append(
                model.wv[category]
            )

    if len(vectors) == 0:
        return np.zeros(
            model.vector_size
        )

    return np.mean(
        vectors,
        axis=0
    )

In [7]:
from node2vec import Node2Vec

node2vec = Node2Vec(
    G,
    dimensions=64,
    walk_length=20,
    num_walks=50,
    workers=4,
    seed=42
)

model = node2vec.fit(
    window=5,
    min_count=1,
    batch_words=128
)
X = np.vstack(
    purchase_df["browser_genre_history"]
    .apply(
        lambda x: history_to_embedding(x, model)
    )
)

print(X.shape)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init="auto"
)

purchase_df["cluster"] = (
    kmeans.fit_predict(X_scaled)
)



Generating walks (CPU: 1): 100%|██████████| 13/13 [00:00<00:00, 81.41it/s]


(717, 64)


In [9]:
purchase_df.head(100)

,user_id,timestamp,genre_path,price,product_id,browser_history,browser_genre_history,cluster
0,486999716,2019-11-01 00:34:18+00:00,electronics.smartphone,242.72,1004768,[1004768],[electronics.smartphone],1
1,512382878,2019-11-01 02:45:00+00:00,electronics.video.tv,228.76,1801805,"[1801805, 1801805]","[electronics.video.tv, electronics.video.tv]",1
2,512400164,2019-11-01 01:50:43+00:00,electronics.smartphone,242.72,1004766,[1004766],[electronics.smartphone],1
3,512405571,2019-11-01 08:00:31+00:00,electronics.smartphone,431.41,1005196,"[28703869, 28703986, 28704348, 28704348, 28703...","[apparel.shoes.keds, apparel.shoes.keds, appar...",1
4,512452645,2019-11-01 02:12:21+00:00,electronics.smartphone,577.89,1003306,[1003306],[electronics.smartphone],1
...,...,...,...,...,...,...,...,...
95,514925962,2019-11-01 07:22:43+00:00,electronics.smartphone,164.93,1004565,"[1005239, 1004565]","[electronics.smartphone, electronics.smartphone]",1
96,514960497,2019-11-01 02:24:25+00:00,electronics.telephone,28.00,8800494,[8800494],[electronics.telephone],1
97,514985745,2019-11-01 04:02:00+00:00,computers.ebooks,123.30,5200131,"[5200131, 5200131, 5200131, 5200131, 5200131, ...","[computers.ebooks, computers.ebooks, computers...",4
98,514986776,2019-11-01 12:59:48+00:00,electronics.tablet,565.75,1201296,"[4803902, 4803902, 1306312, 1306312, 1306312, ...","[electronics.audio.headphone, electronics.audi...",1
